# 🌾 Tripura Crop Data Analysis
## A Farmer-Friendly Guide to Understanding Crop Patterns, Yields & Growing Conditions

**Dataset:** Tripura State — 2955 records | 25 crops | 19 crop years (2004–2023)  
**Goal:** Help farmers understand which crops grow best, under what conditions, and how yields have changed over time.

---

## 📦 Setup — Load Libraries & Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style settings for clear, farmer-friendly charts
plt.rcParams['figure.facecolor'] = '#FAFAF5'
plt.rcParams['axes.facecolor'] = '#FAFAF5'
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

# Load data
df = pd.read_excel('merged_crop_enriched_features.xlsx')

# Extract start year for time series
df['Year'] = df['Crop_Year'].str[:4].astype(int)

print(f"✅ Data loaded: {df.shape[0]} records | {df['Crop'].nunique()} crops | {df['Year'].nunique()} years")
print(f"📅 Year range: {df['Year'].min()} to {df['Year'].max()}")
print(f"🌾 Crops covered: {', '.join(sorted(df['Crop'].unique()))}")

## 📊 Section 1: Data Overview at a Glance

In [ ]:
print("=" * 60)
print("          DATASET SUMMARY")
print("=" * 60)
summary = {
    'Total Records': df.shape[0],
    'Total Crops': df['Crop'].nunique(),
    'Total Seasons': df['Season'].nunique(),
    'Soil Types': df['Soil_Type'].nunique(),
    'Irrigation Methods': df['Irrigation_Type'].nunique(),
    'Avg Yield (T/Ha)': round(df['Yield (Tonne or Bales/Hectare)'].mean(), 2),
    'Avg Rainfall (mm)': round(df['Rainfall_mm'].mean(), 1),
    'Avg Temperature (°C)': round(df['Avg_Temperature_C'].mean(), 1),
    'Avg Fertilizer (kg/ha)': round(df['Fertilizer_kg_per_ha'].mean(), 1),
}
for k, v in summary.items():
    print(f"  {k:<30}: {v}")
print("=" * 60)
df.describe().round(2)

## 🌱 Section 2: Which Crops Are Grown Most in Tripura?
*This shows how many records/seasons each crop appears in — a proxy for how widely grown it is.*

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
crop_counts = df['Crop'].value_counts().sort_values(ascending=True)
colors = plt.cm.YlGn([0.3 + 0.7 * i / len(crop_counts) for i in range(len(crop_counts))])
bars = ax.barh(crop_counts.index, crop_counts.values, color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, crop_counts.values):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)
ax.set_xlabel('Number of Records (Seasons × Years)', fontsize=11)
ax.set_title('🌾 How Many Times Each Crop Was Grown\n(More records = widely grown across years/seasons)', fontsize=13, fontweight='bold', pad=15)
ax.spines[['top','right']].set_visible(False)
ax.set_xlim(0, crop_counts.max() * 1.12)
plt.tight_layout()
plt.savefig('fig1_crop_frequency.png', dpi=150, bbox_inches='tight')
plt.show()
print("👆 Rice, Urad and Moong are the most commonly grown crops in Tripura.")

## 📈 Section 3: Which Crops Give the Best Yield?
*Yield = how much is produced per hectare of land. Higher is better!*

In [ ]:
crop_yield = df.groupby('Crop')['Yield (Tonne or Bales/Hectare)'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
palette = ['#2d6a4f' if y > crop_yield.median() else '#95d5b2' for y in crop_yield.values]
bars = ax.bar(range(len(crop_yield)), crop_yield.values, color=palette, edgecolor='white', linewidth=0.8)

ax.axhline(crop_yield.median(), color='red', linestyle='--', linewidth=1.5, label=f'Overall Median: {crop_yield.median():.2f} T/Ha')
ax.set_xticks(range(len(crop_yield)))
ax.set_xticklabels(crop_yield.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Median Yield (Tonne or Bales per Hectare)', fontsize=11)
ax.set_title('🏆 Crop Yield Comparison\n(Dark green = Above average yield | Light green = Below average)', fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)

for i, (bar, val) in enumerate(zip(bars, crop_yield.values)):
    ax.text(i, val + 0.05, f'{val:.1f}', ha='center', va='bottom', fontsize=8, rotation=0)

plt.tight_layout()
plt.savefig('fig2_yield_by_crop.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"👆 Top 3 yielding crops: {', '.join(crop_yield.head(3).index.tolist())}")
print(f"   Bottom 3 yielding crops: {', '.join(crop_yield.tail(3).index.tolist())}")

## 🗓️ Section 4: Which Season Produces Better Yields?
*Kharif = Monsoon season (June–Nov) | Rabi = Winter season (Nov–Apr)*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Yield by Season boxplot
season_order = df.groupby('Season')['Yield (Tonne or Bales/Hectare)'].median().sort_values(ascending=False).index
season_colors = {'Kharif': '#74c69d', 'Rabi': '#f4a261', 'Autumn': '#e76f51', 'Summer': '#ffd166', 'Winter': '#6d98ba', 'Whole Year': '#9b5de5'}

bp = axes[0].boxplot(
    [df[df['Season'] == s]['Yield (Tonne or Bales/Hectare)'].dropna() for s in season_order],
    labels=season_order, patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch, season in zip(bp['boxes'], season_order):
    patch.set_facecolor(season_colors.get(season, '#cccccc'))
    patch.set_alpha(0.8)
axes[0].set_title('Yield Distribution by Season', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Yield (T or Bales/Ha)')
axes[0].set_xticklabels(season_order, rotation=30, ha='right')
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_ylim(0, 20)  # Focus on main data range
axes[0].text(0.5, -0.18, '(Values above 20 are outliers — removed for clarity)', 
             transform=axes[0].transAxes, ha='center', fontsize=8, color='gray')

# Right: Crop count by season (pie)
season_counts = df['Season'].value_counts()
colors_pie = [season_colors.get(s, '#cccccc') for s in season_counts.index]
wedges, texts, autotexts = axes[1].pie(
    season_counts.values, labels=season_counts.index,
    autopct='%1.1f%%', colors=colors_pie,
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(9)
axes[1].set_title('Share of Records by Season', fontsize=12, fontweight='bold')

plt.suptitle('🗓️ Season-wise Crop Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig3_season_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("👆 Kharif (Monsoon) season has the most crop diversity; whole-year crops tend to show higher yields.")

## 🪱 Section 5: Soil Type & Irrigation — Which Combination Works Best?
*Find out which soil and water method gives the best crop yield*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# 1. Yield by Soil Type
soil_yield = df.groupby('Soil_Type')['Yield (Tonne or Bales/Hectare)'].median().sort_values(ascending=False)
bars1 = axes[0].bar(soil_yield.index, soil_yield.values, color=['#d4a373', '#a2d2ff'], edgecolor='white', linewidth=1.5, width=0.5)
axes[0].set_title('Soil Type vs Median Yield', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Median Yield (T/Ha)')
axes[0].spines[['top','right']].set_visible(False)
for bar, val in zip(bars1, soil_yield.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

# 2. Yield by Irrigation Type
irr_yield = df.groupby('Irrigation_Type')['Yield (Tonne or Bales/Hectare)'].median().sort_values(ascending=False)
irr_colors = {'Canal': '#457b9d', 'Drip': '#a8dadc', 'Rainfed': '#e9c46a'}
bars2 = axes[1].bar(irr_yield.index, irr_yield.values,
    color=[irr_colors.get(i, '#ccc') for i in irr_yield.index],
    edgecolor='white', linewidth=1.5, width=0.5)
axes[1].set_title('Irrigation Method vs Median Yield', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Median Yield (T/Ha)')
axes[1].spines[['top','right']].set_visible(False)
for bar, val in zip(bars2, irr_yield.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

# 3. Heatmap: Soil × Irrigation yield
pivot = df.groupby(['Soil_Type', 'Irrigation_Type'])['Yield (Tonne or Bales/Hectare)'].median().unstack()
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGn', ax=axes[2],
            linewidths=1, linecolor='white', cbar_kws={'label': 'Median Yield'})
axes[2].set_title('Best Soil + Irrigation Combo\n(Darker = Higher Yield)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Irrigation Type')
axes[2].set_ylabel('Soil Type')

plt.suptitle('🪱 Soil & Irrigation Impact on Crop Yield', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4_soil_irrigation.png', dpi=150, bbox_inches='tight')
plt.show()
print("👆 Check the heatmap for the best soil-irrigation combination for your farm!")

## 🐛 Section 6: How Does Pest & Disease Affect Yield?
*Low incidence = few pests, High incidence = many pests attacking crops*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Ordered pest levels
pest_order = ['Low', 'Medium', 'High']
pest_colors = {'Low': '#52b788', 'Medium': '#f4a261', 'High': '#e63946'}

# Left: Violin plot — yield distribution by pest level
pest_data = [df[df['Pest_Disease_Incidence'] == p]['Yield (Tonne or Bales/Hectare)'].clip(upper=15) for p in pest_order]
parts = axes[0].violinplot(pest_data, positions=[1,2,3], showmedians=True, showextrema=False)
for i, (pc, p) in enumerate(zip(parts['bodies'], pest_order)):
    pc.set_facecolor(pest_colors[p])
    pc.set_alpha(0.8)
parts['cmedians'].set_color('black')
parts['cmedians'].set_linewidth(2)
axes[0].set_xticks([1,2,3])
axes[0].set_xticklabels(['🟢 Low\nPest', '🟡 Medium\nPest', '🔴 High\nPest'], fontsize=11)
axes[0].set_ylabel('Yield (T/Ha) — capped at 15 for clarity')
axes[0].set_title('Yield Distribution by Pest Level', fontsize=12, fontweight='bold')
axes[0].spines[['top','right']].set_visible(False)

# Add median labels
for i, p in enumerate(pest_order):
    med = df[df['Pest_Disease_Incidence'] == p]['Yield (Tonne or Bales/Hectare)'].median()
    axes[0].text(i+1, med + 0.3, f'Median:\n{med:.2f}', ha='center', fontsize=9, fontweight='bold')

# Right: Stacked bar showing how pests are distributed across crops (top 8 crops)
top8_crops = df['Crop'].value_counts().head(8).index
pest_crop = df[df['Crop'].isin(top8_crops)].groupby(['Crop', 'Pest_Disease_Incidence']).size().unstack(fill_value=0)
pest_crop = pest_crop.reindex(columns=pest_order, fill_value=0)
pest_crop_pct = pest_crop.div(pest_crop.sum(axis=1), axis=0) * 100
pest_crop_pct.plot(kind='barh', stacked=True, ax=axes[1],
    color=[pest_colors[p] for p in pest_order], edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('% of Records')
axes[1].set_title('Pest Incidence Mix — Top 8 Crops\n(Ideal: mostly green/low pest)', fontsize=12, fontweight='bold')
axes[1].set_xlim(0, 100)
axes[1].legend(title='Pest Level', bbox_to_anchor=(1.01, 1), loc='upper left')
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('🐛 Pest & Disease Impact on Yield', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig5_pest_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("👆 Low pest incidence leads to better yields. High pest attacks significantly reduce production!")

## 🌧️ Section 7: How Do Rainfall, Temperature & Humidity Affect Yield?
*Understanding which weather conditions help crops grow better*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

df_clean = df[df['Yield (Tonne or Bales/Hectare)'] <= 15].copy()  # Remove outliers for clarity

weather_vars = [
    ('Rainfall_mm', '🌧️ Rainfall (mm)', '#1d6fa4'),
    ('Avg_Temperature_C', '🌡️ Temperature (°C)', '#e76f51'),
    ('Humidity_pct', '💧 Humidity (%)', '#48cae4'),
]

for ax, (col, label, color) in zip(axes, weather_vars):
    ax.scatter(df_clean[col], df_clean['Yield (Tonne or Bales/Hectare)'],
               alpha=0.25, s=18, color=color, rasterized=True)
    # Add trend line
    import numpy as np
    z = np.polyfit(df_clean[col].dropna(), df_clean.loc[df_clean[col].notna(), 'Yield (Tonne or Bales/Hectare)'], 1)
    p = np.poly1d(z)
    x_line = sorted(df_clean[col].dropna())
    ax.plot(x_line, p(x_line), color='black', linewidth=2, linestyle='--', label='Trend')
    
    corr = df_clean[col].corr(df_clean['Yield (Tonne or Bales/Hectare)'])
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Yield (T/Ha)', fontsize=10)
    ax.set_title(f'{label}\nvs Yield  |  r = {corr:.2f}', fontsize=11, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    direction = '📈 Higher = Better Yield' if corr > 0.05 else ('📉 Higher = Lower Yield' if corr < -0.05 else '➡️ Weak effect on yield')
    ax.text(0.05, 0.95, direction, transform=ax.transAxes, fontsize=8, va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('🌤️ Weather Conditions vs Crop Yield', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_weather_vs_yield.png', dpi=150, bbox_inches='tight')
plt.show()
print("👆 The trend line (dashed) shows overall direction. r value close to ±1 means strong relationship.")

## 🧪 Section 8: Does More Fertilizer Mean More Yield?
*Finding the sweet spot for fertilizer use by crop*

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

df_clean = df[df['Yield (Tonne or Bales/Hectare)'] <= 15].copy()

# Left: Scatter — Fertilizer vs Yield colored by Irrigation
irr_colors = {'Canal': '#457b9d', 'Drip': '#2dc653', 'Rainfed': '#e9c46a'}
for irr, grp in df_clean.groupby('Irrigation_Type'):
    axes[0].scatter(grp['Fertilizer_kg_per_ha'], grp['Yield (Tonne or Bales/Hectare)'],
                    alpha=0.35, s=20, color=irr_colors.get(irr, '#ccc'), label=irr)
z = np.polyfit(df_clean['Fertilizer_kg_per_ha'], df_clean['Yield (Tonne or Bales/Hectare)'], 1)
p = np.poly1d(z)
x_range = sorted(df_clean['Fertilizer_kg_per_ha'])
axes[0].plot(x_range, p(x_range), 'k--', linewidth=2, label='Overall Trend')
axes[0].set_xlabel('Fertilizer Used (kg per Hectare)', fontsize=11)
axes[0].set_ylabel('Yield (T/Ha)')
axes[0].set_title('Fertilizer vs Yield\n(Colored by Irrigation Type)', fontsize=12, fontweight='bold')
axes[0].legend(title='Irrigation', fontsize=9)
axes[0].spines[['top','right']].set_visible(False)

# Right: Average fertilizer use per crop (top 12)
top12 = df['Crop'].value_counts().head(12).index
fert_crop = df[df['Crop'].isin(top12)].groupby('Crop')['Fertilizer_kg_per_ha'].mean().sort_values()
colors_bar = ['#52b788' if v <= fert_crop.median() else '#e63946' for v in fert_crop.values]
bars = axes[1].barh(fert_crop.index, fert_crop.values, color=colors_bar, edgecolor='white')
axes[1].axvline(fert_crop.median(), color='black', linestyle='--', linewidth=1.5,
               label=f'Median: {fert_crop.median():.0f} kg/ha')
axes[1].set_xlabel('Average Fertilizer (kg per Hectare)', fontsize=11)
axes[1].set_title('Average Fertilizer Used — Top 12 Crops\n(Green = below median | Red = above median)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].spines[['top','right']].set_visible(False)
for bar, val in zip(bars, fert_crop.values):
    axes[1].text(val + 1, bar.get_y() + bar.get_height()/2, f'{val:.0f}', va='center', fontsize=9)

plt.suptitle('🧪 Fertilizer Usage Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig7_fertilizer_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("👆 Too little OR too much fertilizer can hurt. Find the right amount for your crop!")

## 📅 Section 9: How Have Yields Changed Over 19 Years (2004–2023)?
*Track progress — are our farmers getting better yields year after year?*

In [ ]:
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(14, 11))

# Top: Overall yield trend
yearly = df.groupby('Year')['Yield (Tonne or Bales/Hectare)'].agg(['median','mean']).reset_index()
axes[0].fill_between(yearly['Year'],
    df.groupby('Year')['Yield (Tonne or Bales/Hectare)'].quantile(0.25).values,
    df.groupby('Year')['Yield (Tonne or Bales/Hectare)'].quantile(0.75).values,
    alpha=0.2, color='#52b788', label='25th–75th percentile range')
axes[0].plot(yearly['Year'], yearly['median'], 'o-', color='#2d6a4f', linewidth=2.5, markersize=6, label='Median Yield')
axes[0].plot(yearly['Year'], yearly['mean'], 's--', color='#e76f51', linewidth=1.5, markersize=5, label='Mean Yield')
z = np.polyfit(yearly['Year'], yearly['median'], 1)
p = np.poly1d(z)
axes[0].plot(yearly['Year'], p(yearly['Year']), 'b:', linewidth=2, label='Long-term Trend')
axes[0].set_title('📅 Overall Yield Trend — All Crops (2004–2023)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Yield (Tonne or Bales per Hectare)')
axes[0].legend(fontsize=9)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_xticks(yearly['Year'])
axes[0].set_xticklabels(yearly['Year'], rotation=45, fontsize=8)

# Bottom: Top 5 crops yield trend over years
top5_crops = df['Crop'].value_counts().head(5).index
colors5 = ['#2d6a4f', '#e76f51', '#457b9d', '#9b5de5', '#f4a261']
for crop, color in zip(top5_crops, colors5):
    sub = df[df['Crop'] == crop].groupby('Year')['Yield (Tonne or Bales/Hectare)'].median().reset_index()
    axes[1].plot(sub['Year'], sub['Yield (Tonne or Bales/Hectare)'], 'o-', color=color,
                linewidth=2, markersize=5, label=crop)
axes[1].set_title('Top 5 Most-Grown Crops — Yield Over Years', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Median Yield (T/Ha)')
axes[1].set_xlabel('Crop Year')
axes[1].legend(title='Crop', fontsize=9)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_xticks(yearly['Year'])
axes[1].set_xticklabels(yearly['Year'], rotation=45, fontsize=8)

plt.tight_layout()
plt.savefig('fig8_yield_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("👆 An upward blue dotted line means yields have been improving over the years overall!")

## 🌍 Section 10: Land Area vs Production — Which Crops Are Most Productive?
*More land ≠ always more production. This shows efficiency.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: Area vs Production bubble chart (by crop)
crop_summary = df.groupby('Crop').agg(
    area=('Area (Hectare)', 'mean'),
    production=('Production (Tonnes/Bales)', 'mean'),
    yield_=('Yield (Tonne or Bales/Hectare)', 'median')
).reset_index()

scatter = axes[0].scatter(
    crop_summary['area'],
    crop_summary['production'],
    s=crop_summary['yield_'] * 25,
    c=crop_summary['yield_'], cmap='YlGn',
    alpha=0.8, edgecolors='gray', linewidth=0.5
)
plt.colorbar(scatter, ax=axes[0], label='Median Yield (T/Ha)')
for _, row in crop_summary.iterrows():
    if row['area'] > 200 or row['production'] > 500:
        axes[0].annotate(row['Crop'], (row['area'], row['production']),
                         fontsize=7, ha='center', va='bottom', xytext=(0, 5), textcoords='offset points')
axes[0].set_xlabel('Average Area Cultivated (Hectares)')
axes[0].set_ylabel('Average Production (Tonnes/Bales)')
axes[0].set_title('Area vs Production per Crop\n(Bubble size = Yield; Green = High yield)', fontsize=12, fontweight='bold')
axes[0].spines[['top','right']].set_visible(False)

# Right: Total production by crop (bar)
total_prod = df.groupby('Crop')['Production (Tonnes/Bales)'].sum().sort_values(ascending=True).tail(12)
color_bars = plt.cm.YlGn([0.3 + 0.7 * i / len(total_prod) for i in range(len(total_prod))])
axes[1].barh(total_prod.index, total_prod.values / 1000, color=color_bars, edgecolor='white')
axes[1].set_xlabel('Total Production (Thousand Tonnes/Bales)')
axes[1].set_title('Total Production by Crop — All Years\n(Top 12)', fontsize=12, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('🌍 Land Use & Production Efficiency', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig9_area_production.png', dpi=150, bbox_inches='tight')
plt.show()
print("👆 Crops with large bubbles in top-left are most land-efficient — high yield per hectare!")

## 🔥 Section 11: Crop-Season Yield Heatmap — Best Crop for Each Season
*A complete guide for farmers: what to grow and when*

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

pivot_cs = df.groupby(['Crop', 'Season'])['Yield (Tonne or Bales/Hectare)'].median().unstack(fill_value=0)
# Sort crops by overall median yield
crop_order = df.groupby('Crop')['Yield (Tonne or Bales/Hectare)'].median().sort_values(ascending=False).index
pivot_cs = pivot_cs.reindex(crop_order)

sns.heatmap(
    pivot_cs, annot=True, fmt='.1f',
    cmap='YlGn', ax=ax,
    linewidths=0.8, linecolor='white',
    cbar_kws={'label': 'Median Yield (T or Bales/Ha)', 'shrink': 0.7},
    annot_kws={'size': 9}
)

ax.set_title('🌾 CROP × SEASON YIELD GUIDE\nWhat to grow in each season (darker green = higher yield)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Growing Season', fontsize=12)
ax.set_ylabel('Crop', fontsize=12)
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=9)

# Add annotation note
ax.text(1.02, -0.08, '0.0 = Not grown in that season',
        transform=ax.transAxes, fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('fig10_crop_season_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("👆 Use this as a FARMER'S GUIDE — find your season, then pick the darkest green crop for best yield!")

## 🔗 Section 12: What Factors Most Influence Yield?
*Correlation matrix — find the strongest drivers of crop yield*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Correlation matrix
num_cols = ['Area (Hectare)', 'Production (Tonnes/Bales)', 'Rainfall_mm',
            'Avg_Temperature_C', 'Humidity_pct', 'Fertilizer_kg_per_ha',
            'Yield (Tonne or Bales/Hectare)']
corr_matrix = df[num_cols].corr()

# Rename for readability
rename = {
    'Area (Hectare)': 'Area (Ha)',
    'Production (Tonnes/Bales)': 'Production',
    'Rainfall_mm': 'Rainfall',
    'Avg_Temperature_C': 'Temperature',
    'Humidity_pct': 'Humidity',
    'Fertilizer_kg_per_ha': 'Fertilizer',
    'Yield (Tonne or Bales/Hectare)': 'Yield'
}
corr_matrix = corr_matrix.rename(index=rename, columns=rename)

mask = [[i <= j for j in range(len(corr_matrix))] for i in range(len(corr_matrix))]
import numpy as np
mask_arr = np.array(mask)

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=axes[0], linewidths=1, linecolor='white',
            mask=mask_arr, cbar_kws={'shrink': 0.8})
axes[0].set_title('Correlation Between\nAll Numeric Features', fontsize=12, fontweight='bold')

# Right: Horizontal bar of correlations with Yield
yield_corr = corr_matrix['Yield'].drop('Yield').sort_values()
bar_colors = ['#e63946' if v < 0 else '#2d6a4f' for v in yield_corr.values]
bars = axes[1].barh(yield_corr.index, yield_corr.values, color=bar_colors, edgecolor='white', linewidth=0.5)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('Correlation with Yield (r value)')
axes[1].set_title('What Drives Yield?\n(Green = helps yield | Red = hurts yield)', fontsize=12, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)
for bar, val in zip(bars, yield_corr.values):
    offset = 0.01 if val >= 0 else -0.02
    axes[1].text(val + offset, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=10, fontweight='bold')
axes[1].set_xlim(-0.8, 0.8)

plt.suptitle('🔗 Key Factors Affecting Crop Yield', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig11_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("👆 The factor with the highest positive r value is the strongest driver of yield!")

## 📋 Section 13: Farmer's Quick Reference Summary
*Key takeaways from all the analysis above*

In [ ]:
import numpy as np

print("=" * 65)
print("       🌾 FARMER'S QUICK REFERENCE — TRIPURA CROP ANALYSIS")
print("=" * 65)

# Best crop by yield
best_crop = df.groupby('Crop')['Yield (Tonne or Bales/Hectare)'].median().idxmax()
best_yield = df.groupby('Crop')['Yield (Tonne or Bales/Hectare)'].median().max()
print(f"\n🏆 BEST YIELDING CROP   : {best_crop} ({best_yield:.2f} T/Ha median yield)")

# Best season
best_season = df.groupby('Season')['Yield (Tonne or Bales/Hectare)'].median().idxmax()
best_s_yield = df.groupby('Season')['Yield (Tonne or Bales/Hectare)'].median().max()
print(f"🗓️  BEST SEASON          : {best_season} ({best_s_yield:.2f} T/Ha median yield)")

# Best soil
best_soil = df.groupby('Soil_Type')['Yield (Tonne or Bales/Hectare)'].median().idxmax()
print(f"🪱 BEST SOIL TYPE       : {best_soil}")

# Best irrigation
best_irr = df.groupby('Irrigation_Type')['Yield (Tonne or Bales/Hectare)'].median().idxmax()
print(f"💧 BEST IRRIGATION      : {best_irr}")

# Pest impact
low_pest_yield = df[df['Pest_Disease_Incidence']=='Low']['Yield (Tonne or Bales/Hectare)'].median()
high_pest_yield = df[df['Pest_Disease_Incidence']=='High']['Yield (Tonne or Bales/Hectare)'].median()
loss_pct = (low_pest_yield - high_pest_yield) / low_pest_yield * 100
print(f"🐛 PEST IMPACT          : High pest = {loss_pct:.0f}% lower yield than Low pest")

# Fertilizer
corr_fert = df['Fertilizer_kg_per_ha'].corr(df['Yield (Tonne or Bales/Hectare)'])
print(f"🧪 FERTILIZER EFFECT    : Correlation with yield = {corr_fert:.2f} ({'positive' if corr_fert > 0 else 'negative'})")

# Yield trend
first5_yield = df[df['Year'] <= 2008]['Yield (Tonne or Bales/Hectare)'].median()
last5_yield = df[df['Year'] >= 2019]['Yield (Tonne or Bales/Hectare)'].median()
change = (last5_yield - first5_yield) / first5_yield * 100
print(f"📈 YIELD OVER 19 YEARS  : {'+' if change > 0 else ''}{change:.1f}% change (2004-2008 vs 2019-2023)")

print("\n" + "=" * 65)
print("📌 KEY RECOMMENDATIONS FOR FARMERS:")
print(f"  1. Grow {best_crop} for highest yield potential")
print(f"  2. Use {best_irr} irrigation for best results")
print(f"  3. Plant in {best_season} season when possible")
print(f"  4. Use {best_soil} soil fields if available")
print(f"  5. Keep pests LOW — saves up to {loss_pct:.0f}% of your harvest")
print("  6. Balance fertilizer — too much can waste money")
print("=" * 65)

---
## ✅ Analysis Complete!

**Sections covered:**
1. Dataset overview & statistics  
2. Crop frequency (most grown crops)  
3. Yield by crop (best & worst performers)  
4. Season analysis  
5. Soil & irrigation impact  
6. Pest & disease impact  
7. Weather (rainfall, temperature, humidity) vs yield  
8. Fertilizer usage analysis  
9. Yield trends over 19 years  
10. Land use & production efficiency  
11. Crop × season yield heatmap (farmer's planting guide)  
12. Correlation analysis — key yield drivers  
13. Farmer's quick reference summary  

*Analysis focused on Tripura state, India — 2004 to 2023.*